# THEOS-2 Road Surface, Centerline และ Boundary — Dataset Preparation & Fine-tuning

โน้ตบุ๊กนี้เป็นไฟล์ **Colab สำหรับเตรียม weak-label dataset จาก OpenStreetMap (OSM) และฝึกโมเดลจนได้ checkpoint** โดยออกแบบให้ใช้กับภาพ THEOS-2 PMS 4 bands (RGB + NIR) ที่เคยใช้ในโครงการ Building Footprint

## เป้าหมายของงาน

โมเดลทำนายผลพร้อมกัน 3 channel:

1. **Road surface** — พื้นที่ผิวถนน
2. **Road centerline** — แนวกึ่งกลางถนน
3. **Road boundary** — แนวขอบถนนสองด้าน

แนวทางนี้เป็น prototype/รากฐานสำหรับนำไปพัฒนาต่อ โดยใช้ OSM เป็น weak label จึงไม่ใช่ ground truth ที่สมบูรณ์แบบ แต่ช่วยลดเวลาการ digitize ใน QGIS ได้มาก

## Pipeline ภายในโน้ตบุ๊ก

```text
THEOS-2 GeoTIFF
    ↓
ตรวจ CRS / resolution / bands / NoData / valid footprint
    ↓
ดาวน์โหลด OSM highway ตาม footprint ของภาพ
    ↓
ประมาณความกว้างจาก width → lanes → road class
    ↓
สร้าง surface polygon + centerline + boundary mask
    ↓
ตัด patch และแบ่ง Train / Validation แบบ spatial
    ↓
ResNet34 U-Net แบบ 3-output heads
    ↓
ฝึกด้วย BCE + Dice และบันทึก best checkpoint
```

> **หมายเหตุสำคัญ:** ไม่มีระบบใดรับประกันได้ว่า external service เช่น Overpass API จะไม่ล่มเลย โน้ตบุ๊กนี้จึงมี retry, local cache และการใช้ไฟล์ OSM cache เดิมเพื่อป้องกันการเสียเวลารันซ้ำ


## Session 1 — เตรียม Runtime และติดตั้ง Library

ก่อนรัน:

1. ไปที่ **Runtime → Change runtime type**
2. เลือก **GPU**
3. รัน cell ด้านล่าง

โค้ดจะไม่ติดตั้ง PyTorch/torchvision ใหม่ เพื่อหลีกเลี่ยงการทำให้ CUDA ของ Colab เสีย และจะติดตั้งเฉพาะ package ที่ยังไม่มีเท่านั้น


In [ ]:
# ============================================================
# 1 — INSTALL ONLY MISSING PACKAGES
# ============================================================

import sys
import subprocess
import importlib.util

PACKAGE_MAP = {
    "osmnx": "osmnx==2.1.1",
    "rasterio": "rasterio",
    "geopandas": "geopandas",
    "shapely": "shapely",
    "pyogrio": "pyogrio",
    "skimage": "scikit-image",
    "sklearn": "scikit-learn",
    "tqdm": "tqdm",
    "matplotlib": "matplotlib",
    "pandas": "pandas",
}

missing = [pkg for module, pkg in PACKAGE_MAP.items() if importlib.util.find_spec(module) is None]

if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", *missing
    ])
else:
    print("Libraries are already available.")

# ตรวจว่า torch ที่มากับ Colab ยังเป็น CUDA build
import torch
import torchvision

print("Python      :", sys.version.split()[0])
print("PyTorch     :", torch.__version__)
print("Torchvision :", torchvision.__version__)
print("CUDA ready  :", torch.cuda.is_available())


## Session 2 — Mount Google Drive และกำหนด Path กลาง

โดยปกติแก้เฉพาะ cell นี้เมื่อต้องการเปลี่ยนโฟลเดอร์ภาพหรือโฟลเดอร์โครงการ

โครงสร้างที่ระบบจะสร้าง:

```text
Road_Extraction_THEOS2/
├── weak_labels/
│   ├── vector/
│   ├── raster/
│   └── osm_cache/
├── manifests/
├── checkpoints/
├── logs/
└── previews/
```


In [ ]:
# ============================================================
# 2 — DRIVE AND PROJECT PATHS
# ============================================================

from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive")

# ใช้ชุดภาพ THEOS-2 เดิมจากโครงการ Building Footprint
IMAGE_SOURCES = [
    DRIVE_ROOT / "THEOS2_images",
]

# โฟลเดอร์โครงการใหม่
PROJECT_ROOT = (
    DRIVE_ROOT
    / "GeoFM_Projects"
    / "Road_Extraction_THEOS2"
)

VECTOR_LABEL_DIR = PROJECT_ROOT / "weak_labels/vector"
RASTER_LABEL_DIR = PROJECT_ROOT / "weak_labels/raster"
OSM_CACHE_DIR = PROJECT_ROOT / "weak_labels/osm_cache"
MANIFEST_DIR = PROJECT_ROOT / "manifests"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
LOG_DIR = PROJECT_ROOT / "logs"
PREVIEW_DIR = PROJECT_ROOT / "previews"

# Local working directory: อ่าน/เทรนเร็วกว่า Google Drive
LOCAL_WORK_ROOT = Path("/content/theos2_road_work")
LOCAL_IMAGE_DIR = LOCAL_WORK_ROOT / "images"
LOCAL_LABEL_DIR = LOCAL_WORK_ROOT / "labels"

for folder in [
    PROJECT_ROOT, VECTOR_LABEL_DIR, RASTER_LABEL_DIR, OSM_CACHE_DIR,
    MANIFEST_DIR, CHECKPOINT_DIR, LOG_DIR, PREVIEW_DIR,
    LOCAL_WORK_ROOT, LOCAL_IMAGE_DIR, LOCAL_LABEL_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("IMAGE_SOURCES:")
for p in IMAGE_SOURCES:
    print(" -", p)


## Session 3 — Configuration

ค่าเริ่มต้นถูกเลือกให้รันได้ค่อนข้างเร็วบน Colab GPU:

- Patch 512 × 512 pixels = 256 × 256 เมตร เมื่อ GSD = 0.5 เมตร
- ใช้ภาพ 4 bands
- ใช้ OSM highway class ที่รถยนต์ใช้งานได้เป็นหลัก
- จำกัดจำนวน patch เพื่อให้เป็น prototype ที่ฝึกได้ภายในเวลาสมเหตุสมผล
- Validation แยกเชิงพื้นที่ ไม่สุ่ม patch ที่ติดกันทั้งหมด

สามารถเพิ่ม `EPOCHS` และ `MAX_TILES_TOTAL` ภายหลังเมื่อ pipeline ทำงานเรียบร้อยแล้ว


In [ ]:

# ============================================================
# 3 — CENTRAL CONFIGURATION (CENTERLINE V2)
# ============================================================

# ---------- Image ----------
BAND_INDEXES = (1, 2, 3, 4)
RGB_BAND_INDEXES = (1, 2, 3)
NODATA_CANDIDATES = (0, 65535)
VALID_DOWNSAMPLE = 64
VALID_MARGIN_PIXELS = 256

# ---------- OSM / weak labels ----------
FORCE_REFRESH_OSM = False
OSM_QUERY_RETRIES = 4
OSM_RETRY_SECONDS = 15

ALLOWED_HIGHWAY_CLASSES = {
    "motorway", "motorway_link", "trunk", "trunk_link",
    "primary", "primary_link", "secondary", "secondary_link",
    "tertiary", "tertiary_link", "residential", "unclassified",
    "living_street", "service", "road",
}

# ความกว้างรวมโดยประมาณ (เมตร): ใช้เมื่อ OSM ไม่มี width และ lanes
DEFAULT_ROAD_WIDTH_M = {
    "motorway": 14.0, "motorway_link": 7.0,
    "trunk": 12.0, "trunk_link": 7.0,
    "primary": 10.0, "primary_link": 6.5,
    "secondary": 9.0, "secondary_link": 6.0,
    "tertiary": 8.0, "tertiary_link": 5.5,
    "residential": 6.0, "unclassified": 5.5,
    "living_street": 4.5, "service": 4.0, "road": 5.5,
}
LANE_WIDTH_M = 3.2
MIN_ESTIMATED_WIDTH_M = 3.0
MAX_ESTIMATED_WIDTH_M = 20.0

# Centerline V2: train on a soft distance heatmap, not a one/two-pixel binary line.
# sigma controls the Gaussian decay; radius is the maximum supervised support.
CENTERLINE_HEATMAP_SIGMA_M = 1.50
CENTERLINE_HEATMAP_RADIUS_M = 5.00
CENTERLINE_BINARY_WIDTH_M = 1.50       # only for binary preview and validation metrics
CENTERLINE_BINARY_METRIC_THRESHOLD = 0.35
BOUNDARY_WIDTH_PIXELS = 2

# ---------- Dataset ----------
PATCH_SIZE = 512
PATCH_STRIDE = 384
MIN_VALID_RATIO = 0.70
MIN_ROAD_RATIO = 0.002
MIN_CENTERLINE_BINARY_RATIO = 0.00005
NEGATIVE_TO_POSITIVE_RATIO = 0.35
MAX_TILES_TOTAL = 1200                 # use more spatial examples when GPU time permits
VAL_RATIO_IF_SINGLE_IMAGE = 0.20
CENTERLINE_TILE_SAMPLING_WEIGHT = 2.5  # oversample tiles containing a real centerline
RANDOM_SEED = 42
STAGE_VALID_CROPS_TO_LOCAL = True

# ---------- Training ----------
EPOCHS = 30
BATCH_SIZE = 4
NUM_WORKERS = 2
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
USE_AMP = True
USE_IMAGENET_PRETRAINED = True
RESUME_TRAINING = False
RESUME_CHECKPOINT = CHECKPOINT_DIR / "last_theos2_road_multitask_centerline_v2.pth"

# ---------- Loss ----------
HEAD_LOSS_WEIGHTS = {
    "surface": 1.00,
    "centerline": 1.50,
    "boundary": 0.50,
}
CENTERLINE_FOCAL_ALPHA = 0.75
CENTERLINE_FOCAL_GAMMA = 2.0

print("Patch size                    :", PATCH_SIZE)
print("Maximum tiles                 :", MAX_TILES_TOTAL)
print("Epochs                        :", EPOCHS)
print("Centerline heatmap sigma (m)  :", CENTERLINE_HEATMAP_SIGMA_M)
print("Centerline heatmap radius (m) :", CENTERLINE_HEATMAP_RADIUS_M)


## Session 4 — ตรวจ GPU และกำหนด Random Seed

การสร้าง label ใช้ CPU ได้ แต่ขั้นตอน training ต้องใช้ GPU หาก GPU ไม่พร้อม ระบบจะหยุดทันทีเพื่อป้องกันการฝึกบน CPU โดยไม่ตั้งใจ


In [ ]:
# ============================================================
# 4 — REPRODUCIBILITY AND GPU CHECK
# ============================================================

import os
import random
import numpy as np
import torch

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

if not torch.cuda.is_available():
    raise RuntimeError(
        "ไม่พบ GPU กรุณาเลือก Runtime → Change runtime type → GPU "
        "แล้วเริ่มรัน notebook ใหม่"
    )

DEVICE = torch.device("cuda")
torch.backends.cudnn.benchmark = True

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA available:", torch.cuda.is_available())


## Session 5 — ค้นหาและตรวจสอบภาพ THEOS-2

ระบบจะ:

- ค้นหา `.tif`/`.tiff` แบบ recursive
- ตัดไฟล์ที่ชื่อคล้าย mask, prediction หรือ output ออก
- ตรวจจำนวน band, CRS, transform, resolution และ NoData
- หา valid-data bounding box แบบ downsample เพื่อไม่อ่านภาพ 3–5 GB ทั้งไฟล์เข้าหน่วยความจำ


In [ ]:
# ============================================================
# 5 — DISCOVER AND AUDIT THEOS-2 IMAGES
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from rasterio.windows import Window

EXCLUDE_IMAGE_KEYWORDS = {
    "mask", "label", "prob", "prediction", "pred", "clean",
    "centerline", "boundary", "road_surface", "output", "preview",
}


def collect_image_paths(sources):
    paths = []
    for src in sources:
        src = Path(src)
        if not src.exists():
            print("ไม่พบ path และจะข้าม:", src)
            continue
        candidates = [src] if src.is_file() else src.rglob("*")
        for p in candidates:
            if not p.is_file() or p.suffix.lower() not in {".tif", ".tiff"}:
                continue
            name = p.name.lower()
            if any(k in name for k in EXCLUDE_IMAGE_KEYWORDS):
                continue
            paths.append(p)
    return sorted(set(paths), key=lambda p: p.name.lower())


def is_valid_rgb(rgb, nodata_value=None):
    valid = np.any(rgb > 0, axis=0)
    if nodata_value is not None:
        valid &= ~np.all(rgb == nodata_value, axis=0)
    for nd in NODATA_CANDIDATES:
        valid &= ~np.all(rgb == nd, axis=0)
    return valid


def find_valid_bbox_fast(image_path, downsample=64, margin=256):
    with rasterio.open(image_path) as src:
        out_h = max(int(np.ceil(src.height / downsample)), 1)
        out_w = max(int(np.ceil(src.width / downsample)), 1)
        bands = [b for b in RGB_BAND_INDEXES if b <= src.count]
        rgb = src.read(
            bands,
            out_shape=(len(bands), out_h, out_w),
            resampling=Resampling.nearest,
        )
        valid = is_valid_rgb(rgb, src.nodata)
        rows, cols = np.where(valid)
        if len(rows) == 0:
            raise RuntimeError(f"ไม่พบ valid data ใน {Path(image_path).name}")

        sx = src.width / out_w
        sy = src.height / out_h
        left = max(int(np.floor(cols.min() * sx)) - margin, 0)
        right = min(int(np.ceil((cols.max() + 1) * sx)) + margin, src.width)
        top = max(int(np.floor(rows.min() * sy)) - margin, 0)
        bottom = min(int(np.ceil((rows.max() + 1) * sy)) + margin, src.height)
        return (left, top, right, bottom)


def compute_band_percentiles(image_path, bbox, max_size=1000):
    left, top, right, bottom = bbox
    width, height = right - left, bottom - top
    scale = max(width / max_size, height / max_size, 1)
    out_w = max(int(width / scale), 1)
    out_h = max(int(height / scale), 1)

    with rasterio.open(image_path) as src:
        arr = src.read(
            list(BAND_INDEXES),
            window=Window(left, top, width, height),
            out_shape=(len(BAND_INDEXES), out_h, out_w),
            resampling=Resampling.bilinear,
        ).astype(np.float32)
        nodata = src.nodata

    lows, highs = [], []
    for band in arr:
        valid = np.isfinite(band)
        if nodata is not None:
            valid &= band != nodata
        for nd in NODATA_CANDIDATES:
            valid &= band != nd
        values = band[valid]
        if values.size < 100:
            lows.append(0.0)
            highs.append(1.0)
        else:
            lo, hi = np.percentile(values, [2, 98])
            if hi <= lo:
                hi = lo + 1.0
            lows.append(float(lo))
            highs.append(float(hi))
    return lows, highs


IMAGE_PATHS = collect_image_paths(IMAGE_SOURCES)
if not IMAGE_PATHS:
    raise FileNotFoundError(
        "ไม่พบภาพ THEOS-2 กรุณาตรวจ IMAGE_SOURCES และตรวจว่า Google Drive mount แล้ว"
    )

IMAGE_RECORDS = []
for image_path in IMAGE_PATHS:
    with rasterio.open(image_path) as src:
        if src.count < 4:
            raise ValueError(
                f"{image_path.name} มี {src.count} bands แต่ notebook ต้องการ RGB+NIR 4 bands"
            )
        if src.crs is None:
            raise ValueError(f"{image_path.name} ไม่มี CRS")
        bbox = find_valid_bbox_fast(
            image_path, VALID_DOWNSAMPLE, VALID_MARGIN_PIXELS
        )
        lows, highs = compute_band_percentiles(image_path, bbox)
        rec = {
            "image_id": image_path.stem,
            "image_path": str(image_path),
            "width": src.width,
            "height": src.height,
            "count": src.count,
            "dtype": src.dtypes[0],
            "crs": src.crs.to_string(),
            "res_x": abs(src.transform.a),
            "res_y": abs(src.transform.e),
            "nodata": src.nodata,
            "bbox_left": bbox[0], "bbox_top": bbox[1],
            "bbox_right": bbox[2], "bbox_bottom": bbox[3],
            "p02": lows, "p98": highs,
        }
        IMAGE_RECORDS.append(rec)

image_df = pd.DataFrame(IMAGE_RECORDS)
image_audit_csv = MANIFEST_DIR / "image_audit.csv"
image_df.to_csv(image_audit_csv, index=False)

print(f"พบภาพทั้งหมด {len(IMAGE_RECORDS)} ภาพ")
display(image_df[[
    "image_id", "width", "height", "count", "crs", "res_x", "nodata",
    "bbox_left", "bbox_top", "bbox_right", "bbox_bottom"
]])
print("Saved:", image_audit_csv)


## Session 6 — Preview ภาพก่อนสร้าง Label

จุดประสงค์คือยืนยันว่าลำดับ band 1–2–3 แสดงเป็น RGB ถูกต้อง และ valid bounding box ครอบคลุมพื้นที่ภาพจริง

หากสีผิด ให้แก้ `RGB_BAND_INDEXES` และ `BAND_INDEXES` ใน Session 3 ก่อนทำต่อ


In [ ]:
# ============================================================
# 6 — RGB PREVIEW
# ============================================================

import matplotlib.pyplot as plt
from rasterio.windows import Window
from rasterio.enums import Resampling


def normalize_with_percentiles(arr, lows, highs):
    arr = arr.astype(np.float32)
    out = np.zeros_like(arr, dtype=np.float32)
    for i in range(arr.shape[0]):
        out[i] = np.clip((arr[i] - lows[i]) / (highs[i] - lows[i] + 1e-6), 0, 1)
    return out


def read_preview(record, max_size=1000):
    image_path = Path(record["image_path"])
    left, top = int(record["bbox_left"]), int(record["bbox_top"])
    right, bottom = int(record["bbox_right"]), int(record["bbox_bottom"])
    width, height = right - left, bottom - top
    scale = max(width / max_size, height / max_size, 1)
    out_w, out_h = max(int(width / scale), 1), max(int(height / scale), 1)
    with rasterio.open(image_path) as src:
        rgb = src.read(
            list(RGB_BAND_INDEXES),
            window=Window(left, top, width, height),
            out_shape=(3, out_h, out_w),
            resampling=Resampling.bilinear,
        )
    lows = np.array(record["p02"][:3], dtype=np.float32)
    highs = np.array(record["p98"][:3], dtype=np.float32)
    rgb = normalize_with_percentiles(rgb, lows, highs)
    return np.transpose(rgb, (1, 2, 0))

for rec in IMAGE_RECORDS:
    preview = read_preview(rec)
    plt.figure(figsize=(8, 8))
    plt.imshow(preview)
    plt.title(f"{rec['image_id']}\nValid bbox")
    plt.axis("off")
    plt.show()


## Session 7 — ดาวน์โหลด OSM และสร้าง Weak Labels

ระบบจะ query `highway=*` จาก OSM ภายใน valid footprint ของแต่ละภาพ แล้วกรองเฉพาะ road class ที่กำหนด

ลำดับการประมาณความกว้าง:

1. ใช้ tag `width` หากมีและ parse ได้
2. ถ้าไม่มี ใช้ `lanes × 3.2 เมตร`
3. ถ้ายังไม่มี ใช้ค่า default ตาม `highway class`

ผลลัพธ์ vector ต่อภาพ:

- `*_osm_centerline.gpkg`
- `*_osm_surface_buffer.gpkg`

ไฟล์เหล่านี้จะเป็น cache ด้วย หาก Runtime หลุดจะไม่ query OSM ซ้ำเมื่อ `FORCE_REFRESH_OSM=False`


In [ ]:
# ============================================================
# 7 — OSM DOWNLOAD AND VECTOR WEAK-LABEL GENERATION
# ============================================================

import time
import re
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox
from shapely.geometry import box, shape as shapely_shape
from rasterio.warp import transform_geom
from rasterio.windows import Window
from rasterio.windows import bounds as window_bounds

# ตั้งค่าแบบ defensive เพื่อรองรับ OSMnx minor versions
if hasattr(ox.settings, "use_cache"):
    ox.settings.use_cache = True
if hasattr(ox.settings, "cache_folder"):
    ox.settings.cache_folder = str(OSM_CACHE_DIR / "osmnx_http")
if hasattr(ox.settings, "requests_timeout"):
    ox.settings.requests_timeout = 300
if hasattr(ox.settings, "overpass_rate_limit"):
    ox.settings.overpass_rate_limit = True
if hasattr(ox.settings, "log_console"):
    ox.settings.log_console = False


def first_scalar(value):
    if isinstance(value, (list, tuple, set, np.ndarray)):
        return next(iter(value), None)
    return value


def parse_number(value):
    value = first_scalar(value)
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    text = str(value).strip().lower().replace(",", ".")
    match = re.search(r"[-+]?\d*\.?\d+", text)
    if not match:
        return None
    number = float(match.group())
    if "ft" in text or "feet" in text or "foot" in text or "'" in text:
        number *= 0.3048
    return number


def normalize_highway(value):
    values = value if isinstance(value, (list, tuple, set, np.ndarray)) else [value]
    for item in values:
        item = str(item).strip().lower()
        if item in ALLOWED_HIGHWAY_CLASSES:
            return item
    return None


def estimate_width_m(row):
    width = parse_number(row.get("width"))
    if width is not None and width > 0:
        return float(np.clip(width, MIN_ESTIMATED_WIDTH_M, MAX_ESTIMATED_WIDTH_M))

    lanes = parse_number(row.get("lanes"))
    if lanes is not None and lanes > 0:
        estimated = lanes * LANE_WIDTH_M
        return float(np.clip(estimated, MIN_ESTIMATED_WIDTH_M, MAX_ESTIMATED_WIDTH_M))

    highway = row.get("highway_clean")
    estimated = DEFAULT_ROAD_WIDTH_M.get(highway, 5.5)
    return float(np.clip(estimated, MIN_ESTIMATED_WIDTH_M, MAX_ESTIMATED_WIDTH_M))


def valid_footprint_wgs84(record):
    image_path = Path(record["image_path"])
    left, top = int(record["bbox_left"]), int(record["bbox_top"])
    right, bottom = int(record["bbox_right"]), int(record["bbox_bottom"])
    with rasterio.open(image_path) as src:
        win = Window(left, top, right - left, bottom - top)
        b = window_bounds(win, src.transform)
        geom_src = box(*b)
        geom_wgs84 = transform_geom(
            src.crs, "EPSG:4326", geom_src.__geo_interface__, precision=8
        )
    return shapely_shape(geom_wgs84)


def download_osm_features_with_retry(polygon_wgs84):
    last_error = None
    for attempt in range(1, OSM_QUERY_RETRIES + 1):
        try:
            print(f"OSM query attempt {attempt}/{OSM_QUERY_RETRIES}")
            return ox.features.features_from_polygon(
                polygon_wgs84,
                tags={"highway": True},
            )
        except Exception as exc:
            last_error = exc
            print("OSM query failed:", repr(exc))
            if attempt < OSM_QUERY_RETRIES:
                wait = OSM_RETRY_SECONDS * attempt
                print(f"Waiting {wait} seconds before retry...")
                time.sleep(wait)
    raise RuntimeError(
        "ดาวน์โหลด OSM ไม่สำเร็จหลัง retry ครบแล้ว "
        "ให้ลองใหม่ภายหลัง หรือวาง GeoPackage cache ไว้ใน OSM_CACHE_DIR"
    ) from last_error


def prepare_osm_vectors(record):
    image_id = record["image_id"]
    center_path = VECTOR_LABEL_DIR / f"{image_id}_osm_centerline.gpkg"
    surface_path = VECTOR_LABEL_DIR / f"{image_id}_osm_surface_buffer.gpkg"

    if center_path.exists() and surface_path.exists() and not FORCE_REFRESH_OSM:
        print("Using cached vectors:", image_id)
        try:
            center = gpd.read_file(center_path, layer="road_centerline")
            surface = gpd.read_file(surface_path, layer="road_surface")
            if len(center) > 0 and len(surface) > 0:
                return center, surface, center_path, surface_path
            print("Cached file is empty; downloading again.")
        except Exception as exc:
            print("Cached vector is incomplete/corrupted; rebuilding:", repr(exc))

    polygon_wgs84 = valid_footprint_wgs84(record)
    raw = download_osm_features_with_retry(polygon_wgs84)

    if raw is None or len(raw) == 0:
        raise RuntimeError(f"OSM ไม่คืนถนนสำหรับ {image_id}")

    roads = raw.reset_index().copy()
    roads = roads[roads.geometry.notna() & ~roads.geometry.is_empty].copy()
    roads = roads[roads.geometry.geom_type.isin(["LineString", "MultiLineString"])].copy()
    roads = roads.explode(index_parts=False, ignore_index=True)
    roads["highway_clean"] = roads.get("highway", pd.Series(index=roads.index, dtype=object)).apply(normalize_highway)
    roads = roads[roads["highway_clean"].notna()].copy()

    if len(roads) == 0:
        raise RuntimeError(f"ไม่เหลือ road class ที่อนุญาตสำหรับ {image_id}")

    image_path = Path(record["image_path"])
    with rasterio.open(image_path) as src:
        target_crs = src.crs
        if target_crs is None:
            raise RuntimeError(f"{image_id}: raster has no CRS; metre-based OSM buffers are unsafe")
        if target_crs.is_geographic:
            raise RuntimeError(
                f"{image_id}: raster CRS is geographic ({target_crs}). "
                "Reproject the THEOS-2 raster to a projected CRS in metres before creating OSM buffers."
            )
        left, top = int(record["bbox_left"]), int(record["bbox_top"])
        right, bottom = int(record["bbox_right"]), int(record["bbox_bottom"])
        valid_bounds = window_bounds(Window(left, top, right-left, bottom-top), src.transform)
        clip_polygon = box(*valid_bounds)

    roads = roads.to_crs(target_crs)
    roads = roads[roads.intersects(clip_polygon)].copy()
    roads["geometry"] = roads.geometry.intersection(clip_polygon)
    roads = roads[roads.geometry.notna() & ~roads.geometry.is_empty].copy()
    roads = roads.explode(index_parts=False, ignore_index=True)
    roads = roads[roads.geometry.geom_type == "LineString"].copy()

    # target_crs is validated as projected above, therefore width and buffer units are metres.
    roads["estimated_width_m"] = roads.apply(estimate_width_m, axis=1)
    roads["length_m"] = roads.geometry.length
    roads = roads[roads["length_m"] >= 2.0].copy()

    # ลบ geometry ซ้ำเพื่อไม่ให้ mask หนาเกินจาก duplicate OSM objects
    roads["geom_key"] = roads.geometry.apply(lambda g: g.wkb_hex)
    roads = roads.drop_duplicates(subset=["geom_key"]).drop(columns=["geom_key"])

    keep_cols = [
        c for c in ["element", "id", "osmid", "name", "highway_clean",
                    "lanes", "width", "surface", "estimated_width_m", "length_m", "geometry"]
        if c in roads.columns
    ]
    center = roads[keep_cols].copy()

    # GeoPackage ไม่รองรับ Python list/object ซับซ้อนโดยตรง จึงแปลงเป็นข้อความ
    def to_gpkg_scalar(value):
        if isinstance(value, (list, tuple, set, np.ndarray)):
            return ";".join(map(str, value))
        if value is None or (isinstance(value, float) and np.isnan(value)):
            return None
        return value

    for column in center.columns:
        if column != "geometry" and center[column].dtype == "object":
            center[column] = center[column].apply(to_gpkg_scalar)

    surface_geoms = [
        geom.buffer(float(width) / 2.0, cap_style=2, join_style=2)
        for geom, width in zip(center.geometry, center["estimated_width_m"])
    ]
    surface = center.drop(columns="geometry").copy()
    surface = gpd.GeoDataFrame(surface, geometry=surface_geoms, crs=center.crs)
    surface = surface[surface.geometry.notna() & ~surface.geometry.is_empty].copy()

    for p in [center_path, surface_path]:
        if p.exists():
            p.unlink()

    center.to_file(center_path, driver="GPKG", layer="road_centerline", index=False)
    surface.to_file(surface_path, driver="GPKG", layer="road_surface", index=False)

    return center, surface, center_path, surface_path


VECTOR_RECORDS = []
for rec in IMAGE_RECORDS:
    print("\n" + "=" * 100)
    print("Preparing OSM weak labels:", rec["image_id"])
    center, surface, center_path, surface_path = prepare_osm_vectors(rec)
    VECTOR_RECORDS.append({
        "image_id": rec["image_id"],
        "center_path": str(center_path),
        "surface_path": str(surface_path),
        "road_lines": len(center),
        "surface_parts": len(surface),
        "total_centerline_km": float(center.geometry.length.sum() / 1000.0),
    })
    print("Road lines:", len(center))
    print("Centerline length (km):", center.geometry.length.sum() / 1000.0)

vector_df = pd.DataFrame(VECTOR_RECORDS)
vector_csv = MANIFEST_DIR / "osm_vector_summary.csv"
vector_df.to_csv(vector_csv, index=False)
display(vector_df)
print("Saved:", vector_csv)


## Session 8 — Rasterize เป็น Surface / Centerline / Boundary Masks

Mask ถูกสร้างเฉพาะ valid crop แทนการสร้างเต็มภาพ 25,000 × 25,000 pixels จึงประหยัดพื้นที่และอ่านเร็วกว่า

- Surface mask มาจาก OSM centerline buffer
- Centerline mask มาจาก OSM line และขยายความหนาเล็กน้อยเพื่อให้โมเดลเรียนง่าย
- Boundary mask มาจากขอบด้านในของ surface mask

ทุก mask มี CRS และ transform ตรงกับ valid crop ของภาพต้นฉบับ


In [ ]:

# ============================================================
# 8 — RASTERIZE MULTI-TASK LABELS (CENTERLINE V2)
# ============================================================

import shutil
from scipy.ndimage import binary_dilation, binary_erosion, distance_transform_edt
from rasterio.features import rasterize
from rasterio.windows import transform as window_transform


def safe_raster_profile(src, width, height, transform, count=1, dtype="uint8", nodata=0):
    profile = src.profile.copy()
    profile.update(
        width=int(width), height=int(height), transform=transform,
        count=count, dtype=dtype, nodata=nodata,
        compress="deflate", predictor=2 if dtype != "uint8" else 1,
        tiled=True, blockxsize=512, blockysize=512, BIGTIFF="IF_SAFER",
    )
    return profile


def write_mask(path, array, profile):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(array.astype(profile["dtype"]), 1)


def centerline_heatmap_from_osm(center_core, surface_mask, res_x_m, res_y_m):
    """Gaussian distance target in metres, limited to road surface.

    The original OSM line has distance zero and receives value 1.0.  The target
    fades smoothly to zero at CENTERLINE_HEATMAP_RADIUS_M, making small OSM/image
    registration errors learnable instead of counting as a complete miss.
    """
    if not center_core.any():
        return np.zeros(center_core.shape, dtype=np.float32), np.zeros(center_core.shape, dtype=bool)

    distance_m = distance_transform_edt(
        ~center_core,
        sampling=(float(res_y_m), float(res_x_m)),
    ).astype(np.float32)
    sigma = max(float(CENTERLINE_HEATMAP_SIGMA_M), min(res_x_m, res_y_m) / 2.0)
    heatmap = np.exp(-0.5 * (distance_m / sigma) ** 2).astype(np.float32)
    heatmap[distance_m > float(CENTERLINE_HEATMAP_RADIUS_M)] = 0.0
    heatmap[~surface_mask] = 0.0

    binary_radius_m = max(float(CENTERLINE_BINARY_WIDTH_M) / 2.0, min(res_x_m, res_y_m) / 2.0)
    binary = (distance_m <= binary_radius_m) & surface_mask
    return heatmap, binary


def rasterize_labels_for_record(record):
    image_id = record["image_id"]
    image_path = Path(record["image_path"])
    center_path = VECTOR_LABEL_DIR / f"{image_id}_osm_centerline.gpkg"
    surface_path = VECTOR_LABEL_DIR / f"{image_id}_osm_surface_buffer.gpkg"

    center = gpd.read_file(center_path, layer="road_centerline")
    surface = gpd.read_file(surface_path, layer="road_surface")

    left, top = int(record["bbox_left"]), int(record["bbox_top"])
    right, bottom = int(record["bbox_right"]), int(record["bbox_bottom"])
    crop_w, crop_h = right - left, bottom - top

    with rasterio.open(image_path) as src:
        if src.crs is None or src.crs.is_geographic:
            raise RuntimeError(
                f"{image_id}: labels require a projected CRS in metres; got {src.crs}."
            )
        win = Window(left, top, crop_w, crop_h)
        crop_transform = window_transform(win, src.transform)
        surface_profile = safe_raster_profile(src, crop_w, crop_h, crop_transform, dtype="uint8", nodata=0)
        heatmap_profile = safe_raster_profile(src, crop_w, crop_h, crop_transform, dtype="float32", nodata=0.0)
        res_x_m, res_y_m = abs(float(src.transform.a)), abs(float(src.transform.e))

    surface_mask = rasterize(
        [(geom, 1) for geom in surface.geometry if geom is not None and not geom.is_empty],
        out_shape=(crop_h, crop_w), transform=crop_transform, fill=0,
        all_touched=True, dtype="uint8",
    ).astype(bool)

    # This is the OSM centre geometry before any artificial width is applied.
    center_core = rasterize(
        [(geom, 1) for geom in center.geometry if geom is not None and not geom.is_empty],
        out_shape=(crop_h, crop_w), transform=crop_transform, fill=0,
        all_touched=True, dtype="uint8",
    ).astype(bool)
    centerline_heatmap, centerline_binary = centerline_heatmap_from_osm(
        center_core, surface_mask, res_x_m, res_y_m
    )

    eroded = binary_erosion(
        surface_mask, iterations=max(int(BOUNDARY_WIDTH_PIXELS), 1), border_value=0,
    )
    boundary_mask = surface_mask & ~eroded

    out_paths = {
        "surface": RASTER_LABEL_DIR / f"{image_id}_road_surface_mask.tif",
        "centerline": RASTER_LABEL_DIR / f"{image_id}_road_centerline_heatmap.tif",
        "centerline_binary": RASTER_LABEL_DIR / f"{image_id}_road_centerline_binary_mask.tif",
        "boundary": RASTER_LABEL_DIR / f"{image_id}_road_boundary_mask.tif",
    }
    write_mask(out_paths["surface"], surface_mask.astype(np.uint8), surface_profile)
    write_mask(out_paths["centerline"], centerline_heatmap, heatmap_profile)
    write_mask(out_paths["centerline_binary"], centerline_binary.astype(np.uint8), surface_profile)
    write_mask(out_paths["boundary"], boundary_mask.astype(np.uint8), surface_profile)

    return {
        "image_id": image_id,
        "crop_left": left, "crop_top": top,
        "crop_width": crop_w, "crop_height": crop_h,
        "surface_path": str(out_paths["surface"]),
        "centerline_path": str(out_paths["centerline"]),
        "centerline_binary_path": str(out_paths["centerline_binary"]),
        "boundary_path": str(out_paths["boundary"]),
        "surface_ratio": float(surface_mask.mean()),
        "centerline_heatmap_mean": float(centerline_heatmap.mean()),
        "centerline_binary_ratio": float(centerline_binary.mean()),
        "boundary_ratio": float(boundary_mask.mean()),
        "resolution_x_m": res_x_m,
        "resolution_y_m": res_y_m,
    }


LABEL_RECORDS = []
for rec in IMAGE_RECORDS:
    print("Rasterizing Centerline V2 labels:", rec["image_id"])
    LABEL_RECORDS.append(rasterize_labels_for_record(rec))

label_df = pd.DataFrame(LABEL_RECORDS)
label_summary_csv = MANIFEST_DIR / "label_raster_summary_centerline_v2.csv"
label_df.to_csv(label_summary_csv, index=False)
display(label_df)
print("Saved:", label_summary_csv)


## Session 9 — ตรวจ Overlay ของ Weak Labels

ให้ดูอย่างน้อย 2–3 ภาพก่อนฝึก:

- เส้น OSM ควรทับตำแหน่งถนนโดยประมาณ
- surface buffer ไม่ควรกว้างหรือแคบผิดอย่างรุนแรง
- offset เล็กน้อยยอมรับได้สำหรับ prototype แต่ถ้า offset หลายเมตรทุกเส้น ควรหยุดและตรวจ CRS/orthorectification ก่อน


In [ ]:
# ============================================================
# 9 — LABEL OVERLAY PREVIEW (CENTERLINE V2)
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
from scipy.ndimage import maximum_filter
from rasterio.enums import Resampling


def read_label_preview(mask_path, max_size=1000, resampling=Resampling.nearest):
    with rasterio.open(mask_path) as src:
        scale = max(src.width / max_size, src.height / max_size, 1)
        out_w = max(int(src.width / scale), 1)
        out_h = max(int(src.height / scale), 1)

        return src.read(
            1,
            out_shape=(out_h, out_w),
            resampling=resampling,
        )


def thicken_for_preview(binary_mask, pixels=1):
    """
    ขยายเฉพาะภาพ preview เพื่อให้เส้น 1-pixel ไม่หายตอนย่อภาพ
    ไม่ได้แก้ไข label GeoTIFF จริง
    """
    binary_mask = binary_mask.astype(bool)

    if pixels <= 0:
        return binary_mask

    size = 2 * int(pixels) + 1
    return maximum_filter(
        binary_mask.astype(np.uint8),
        size=size,
        mode="constant",
    ) > 0


for rec, lab in zip(IMAGE_RECORDS, LABEL_RECORDS):
    rgb = read_preview(rec, max_size=1000)

    # ใช้ nearest กับ rasterio.read ได้แน่นอน
    surface = read_label_preview(
        lab["surface_path"],
        resampling=Resampling.nearest,
    ) > 0

    center_binary = read_label_preview(
        lab["centerline_binary_path"],
        resampling=Resampling.nearest,
    ) > 0

    # Heatmap เป็น float ใช้ bilinear ได้
    center_heatmap = read_label_preview(
        lab["centerline_path"],
        resampling=Resampling.bilinear,
    )

    boundary = read_label_preview(
        lab["boundary_path"],
        resampling=Resampling.nearest,
    ) > 0

    # ขยายเฉพาะเส้นตอนแสดงผล
    center_display = thicken_for_preview(center_binary, pixels=1)
    boundary_display = thicken_for_preview(boundary, pixels=1)

    overlay = rgb.copy()
    overlay[surface] = (
        0.65 * overlay[surface] +
        0.35 * np.array([1.0, 0.0, 1.0])
    )
    overlay[boundary_display] = np.array([1.0, 0.8, 0.0])
    overlay[center_display] = np.array([0.0, 1.0, 1.0])

    fig, axes = plt.subplots(1, 5, figsize=(25, 5))

    axes[0].imshow(rgb)
    axes[0].set_title("THEOS-2 RGB")

    axes[1].imshow(surface, cmap="gray")
    axes[1].set_title("Label: surface")

    axes[2].imshow(center_heatmap, cmap="magma", vmin=0, vmax=1)
    axes[2].set_title("Target: centerline heatmap")

    axes[3].imshow(center_display, cmap="gray")
    axes[3].set_title("Metric label: centerline binary")

    axes[4].imshow(overlay)
    axes[4].set_title("Surface + boundary + centerline")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

## Session 10 — Stage Valid Crops ไปยัง Local Disk

Google Drive เหมาะกับเก็บไฟล์ถาวร แต่ random window reading ระหว่าง training ช้ากว่า local disk มาก

Cell นี้จะ copy เฉพาะ valid crop ของแต่ละภาพและ mask ไป `/content` โดยอ่านแบบ block จึงไม่กิน RAM ทั้งภาพพร้อมกัน


In [ ]:

# ============================================================
# 10 — STAGE VALID CROPS AND LABELS TO /content (CENTERLINE V2)
# ============================================================

import shutil
from tqdm.auto import tqdm
from rasterio.windows import Window
from rasterio.windows import transform as window_transform


def copy_crop_blockwise(src_path, dst_path, bbox, band_indexes):
    src_path, dst_path = Path(src_path), Path(dst_path)
    left, top, right, bottom = map(int, bbox)
    width, height = right - left, bottom - top
    with rasterio.open(src_path) as src:
        crop_win = Window(left, top, width, height)
        profile = src.profile.copy()
        profile.update(width=width, height=height, transform=window_transform(crop_win, src.transform),
                       count=len(band_indexes), compress="deflate", predictor=2, tiled=True,
                       blockxsize=512, blockysize=512, BIGTIFF="IF_SAFER")
        if dst_path.exists():
            dst_path.unlink()
        with rasterio.open(dst_path, "w", **profile) as dst:
            for y in tqdm(range(0, height, 1024), desc=f"Stage {src_path.stem}"):
                h = min(1024, height-y)
                for x in range(0, width, 1024):
                    w = min(1024, width-x)
                    dst.write(src.read(list(band_indexes), window=Window(left+x, top+y, w, h)), window=Window(x, y, w, h))
    return dst_path


TRAINING_RECORDS = []
for rec, lab in zip(IMAGE_RECORDS, LABEL_RECORDS):
    image_id = rec["image_id"]
    bbox = (rec["bbox_left"], rec["bbox_top"], rec["bbox_right"], rec["bbox_bottom"])
    local_image = LOCAL_IMAGE_DIR / f"{image_id}_valid_crop.tif"
    if not STAGE_VALID_CROPS_TO_LOCAL or not local_image.exists():
        copy_crop_blockwise(rec["image_path"], local_image, bbox, BAND_INDEXES)
    else:
        print("Local image already exists:", local_image)

    local_paths = {}
    for key in ["surface", "centerline", "centerline_binary", "boundary"]:
        src_label = Path(lab[f"{key}_path"])
        dst_label = LOCAL_LABEL_DIR / src_label.name
        shutil.copy2(src_label, dst_label)
        local_paths[key] = dst_label
    TRAINING_RECORDS.append({
        "image_id": image_id, "image_path": str(local_image),
        "surface_path": str(local_paths["surface"]),
        "centerline_path": str(local_paths["centerline"]),
        "centerline_binary_path": str(local_paths["centerline_binary"]),
        "boundary_path": str(local_paths["boundary"]),
        "p02": rec["p02"], "p98": rec["p98"],
    })

print("Local staged records:", len(TRAINING_RECORDS))
for rec in TRAINING_RECORDS:
    print(" -", rec["image_id"], rec["image_path"])


## Session 11 — สร้าง Tile Manifest และ Spatial Split

ระบบคัดเลือก patch โดย:

- valid pixel ≥ `MIN_VALID_RATIO`
- เก็บ patch ที่มี road surface ≥ `MIN_ROAD_RATIO`
- เพิ่ม negative patch จำนวนหนึ่งเพื่อให้โมเดลแยกพื้นที่ไม่ใช่ถนน
- ถ้ามีหลายภาพ จะกันภาพสุดท้ายเป็น validation เพื่อทดสอบข้ามพื้นที่
- ถ้ามีภาพเดียว จะกัน spatial blocks บางส่วนเป็น validation

Manifest เป็น CSV จึงตรวจสอบและนำกลับมาใช้ซ้ำได้


In [ ]:

# ============================================================
# 11 — BUILD TILE MANIFEST (CENTERLINE V2)
# ============================================================

import hashlib
import random
from rasterio.windows import Window


def positions_for_length(length, patch, stride):
    if length <= patch:
        return [0]
    positions = list(range(0, length - patch + 1, stride))
    last = length - patch
    if positions[-1] != last:
        positions.append(last)
    return positions


def patch_valid_ratio(image_src, win):
    arr = image_src.read([1, 2, 3], window=win)
    valid = is_valid_rgb(arr, image_src.nodata)
    return float(valid.mean())


def spatial_val_assignment(image_id, x, y, ratio=0.20):
    key = f"{image_id}:{x//PATCH_SIZE}:{y//PATCH_SIZE}".encode()
    value = int(hashlib.md5(key).hexdigest()[:8], 16) / 0xFFFFFFFF
    return value < ratio


all_rows = []
for record in TRAINING_RECORDS:
    image_id = record["image_id"]
    with (
        rasterio.open(record["image_path"]) as img_src,
        rasterio.open(record["surface_path"]) as surf_src,
        rasterio.open(record["centerline_binary_path"]) as center_src,
    ):
        if img_src.width != surf_src.width or img_src.height != surf_src.height:
            raise RuntimeError(f"Local image/label size mismatch: {image_id}")
        if img_src.width != center_src.width or img_src.height != center_src.height:
            raise RuntimeError(f"Local image/centerline label size mismatch: {image_id}")

        xs = positions_for_length(img_src.width, PATCH_SIZE, PATCH_STRIDE)
        ys = positions_for_length(img_src.height, PATCH_SIZE, PATCH_STRIDE)
        for y in tqdm(ys, desc=f"Manifest {image_id}"):
            for x in xs:
                win = Window(x, y, PATCH_SIZE, PATCH_SIZE)
                valid_ratio = patch_valid_ratio(img_src, win)
                if valid_ratio < MIN_VALID_RATIO:
                    continue
                surface = surf_src.read(1, window=win)
                center_binary = center_src.read(1, window=win)
                road_ratio = float((surface > 0).mean())
                centerline_ratio = float((center_binary > 0).mean())
                all_rows.append({
                    "image_id": image_id,
                    "image_path": record["image_path"],
                    "surface_path": record["surface_path"],
                    "centerline_path": record["centerline_path"],
                    "centerline_binary_path": record["centerline_binary_path"],
                    "boundary_path": record["boundary_path"],
                    "x": x, "y": y, "patch_size": PATCH_SIZE,
                    "valid_ratio": valid_ratio, "road_ratio": road_ratio,
                    "centerline_ratio": centerline_ratio,
                    "is_positive": road_ratio >= MIN_ROAD_RATIO,
                    "is_centerline_positive": centerline_ratio >= MIN_CENTERLINE_BINARY_RATIO,
                    "p02": json.dumps(record["p02"]), "p98": json.dumps(record["p98"]),
                })

manifest = pd.DataFrame(all_rows)
if len(manifest) == 0:
    raise RuntimeError("ไม่พบ patch ที่ valid ตรวจ valid bbox, NoData และ PATCH_SIZE")

# Never lose a tile containing a binary OSM centreline due to negative subsampling.
keep = manifest[manifest["is_positive"] | manifest["is_centerline_positive"]].copy()
negative = manifest[~(manifest["is_positive"] | manifest["is_centerline_positive"])].copy()
if len(keep) == 0:
    raise RuntimeError("ไม่พบ road/centerline positive tile ตรวจ OSM overlay และ labels")
max_negative = int(round(len(keep) * NEGATIVE_TO_POSITIVE_RATIO))
if len(negative) > max_negative:
    negative = negative.sample(max_negative, random_state=RANDOM_SEED)
manifest = pd.concat([keep, negative], ignore_index=True).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

if len(manifest) > MAX_TILES_TOTAL:
    # Preserve every centreline-positive tile first; only downsample remaining tiles if necessary.
    center_keep = manifest[manifest["is_centerline_positive"]].copy()
    other = manifest[~manifest["is_centerline_positive"]].copy()
    if len(center_keep) >= MAX_TILES_TOTAL:
        manifest = center_keep.sample(MAX_TILES_TOTAL, random_state=RANDOM_SEED)
        print("Warning: centerline-positive tiles exceed MAX_TILES_TOTAL; sampled only centreline tiles.")
    else:
        other_n = MAX_TILES_TOTAL - len(center_keep)
        manifest = pd.concat([center_keep, other.sample(min(other_n, len(other)), random_state=RANDOM_SEED)], ignore_index=True)
    manifest = manifest.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

unique_images = sorted(manifest["image_id"].unique())
if len(unique_images) >= 2:
    validation_image = unique_images[-1]
    manifest["split"] = np.where(manifest["image_id"] == validation_image, "val", "train")
    print("Validation image:", validation_image)
else:
    manifest["split"] = [
        "val" if spatial_val_assignment(row.image_id, row.x, row.y, VAL_RATIO_IF_SINGLE_IMAGE) else "train"
        for row in manifest.itertuples()
    ]

if (manifest["split"] == "train").sum() == 0 or (manifest["split"] == "val").sum() == 0:
    manifest = manifest.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    cut = max(1, int(len(manifest) * (1 - VAL_RATIO_IF_SINGLE_IMAGE)))
    manifest["split"] = "val"
    manifest.loc[:cut-1, "split"] = "train"

manifest_path = MANIFEST_DIR / "road_multitask_tile_manifest_centerline_v2.csv"
manifest.to_csv(manifest_path, index=False)
print("Total tiles:", len(manifest))
print(manifest.groupby(["split", "is_centerline_positive"]).size())
print("Saved:", manifest_path)
display(manifest.head())


## Session 12 — PyTorch Dataset, Augmentation และ DataLoader

Augmentation ใช้เฉพาะ transform ที่ไม่ทำลายความหมายของความกว้างถนน:

- flip
- หมุน 90 องศา
- brightness/contrast/gamma เล็กน้อย

ไม่ใช้ non-uniform scaling ที่แรง เพราะจะทำให้ width label เปลี่ยนรูป


In [ ]:

# ============================================================
# 12 — DATASET AND DATALOADER (CENTERLINE V2)
# ============================================================

import json
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler


class RoadMultiTaskDataset(Dataset):
    def __init__(self, frame, train=True):
        self.frame = frame.reset_index(drop=True).copy()
        self.train = train

    def __len__(self):
        return len(self.frame)

    @staticmethod
    def _read_window(path, indexes, x, y, size):
        with rasterio.open(path) as src:
            return src.read(indexes, window=Window(x, y, size, size))

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        x, y, size = int(row.x), int(row.y), int(row.patch_size)
        image = self._read_window(row.image_path, list(range(1, len(BAND_INDEXES)+1)), x, y, size).astype(np.float32)
        surface = self._read_window(row.surface_path, [1], x, y, size)[0]
        centerline_heatmap = self._read_window(row.centerline_path, [1], x, y, size)[0]
        centerline_binary = self._read_window(row.centerline_binary_path, [1], x, y, size)[0]
        boundary = self._read_window(row.boundary_path, [1], x, y, size)[0]
        targets = np.stack([surface, centerline_heatmap, boundary], axis=0).astype(np.float32)
        centerline_binary = (centerline_binary > 0).astype(np.float32)[None, ...]

        lows = np.asarray(json.loads(row.p02), dtype=np.float32)[:, None, None]
        highs = np.asarray(json.loads(row.p98), dtype=np.float32)[:, None, None]
        image = np.clip((image - lows) / (highs - lows + 1e-6), 0, 1)
        image[~np.isfinite(image)] = 0

        if self.train:
            if np.random.rand() < 0.5:
                image, targets, centerline_binary = image[:, :, ::-1], targets[:, :, ::-1], centerline_binary[:, :, ::-1]
            if np.random.rand() < 0.5:
                image, targets, centerline_binary = image[:, ::-1, :], targets[:, ::-1, :], centerline_binary[:, ::-1, :]
            k = np.random.randint(0, 4)
            if k:
                image = np.rot90(image, k, axes=(1, 2))
                targets = np.rot90(targets, k, axes=(1, 2))
                centerline_binary = np.rot90(centerline_binary, k, axes=(1, 2))
            if np.random.rand() < 0.7:
                image = np.clip(image * np.random.uniform(0.90, 1.10) + np.random.uniform(-0.06, 0.06), 0, 1)
            if np.random.rand() < 0.3:
                image = np.clip(image, 0, 1) ** np.random.uniform(0.90, 1.10)

        return (
            torch.from_numpy(np.ascontiguousarray(image, dtype=np.float32)),
            torch.from_numpy(np.ascontiguousarray(targets, dtype=np.float32)),
            torch.from_numpy(np.ascontiguousarray(centerline_binary, dtype=np.float32)),
        )


train_df = manifest[manifest["split"] == "train"].reset_index(drop=True)
val_df = manifest[manifest["split"] == "val"].reset_index(drop=True)
train_dataset = RoadMultiTaskDataset(train_df, train=True)
val_dataset = RoadMultiTaskDataset(val_df, train=False)

sample_weights = np.where(
    train_df["is_centerline_positive"].to_numpy(),
    float(CENTERLINE_TILE_SAMPLING_WEIGHT), 1.0,
).astype(np.float64)
train_sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, sampler=train_sampler,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0,
)

images, targets, centerline_binary = next(iter(train_loader))
print("Train tiles:", len(train_dataset), "| Val tiles:", len(val_dataset))
print("Image batch:", tuple(images.shape), "| Target batch:", tuple(targets.shape))
print("Binary centerline batch:", tuple(centerline_binary.shape))


## Session 13 — โมเดล ResNet34 U-Net แบบ Multi-task

เลือก ResNet34 U-Net แทน TerraMind ในเวอร์ชันแรก เพราะ:

- dependency น้อยกว่า
- ใช้ pretrained ImageNet ได้
- รองรับ 4 bands โดยเพิ่ม NIR เข้า first convolution
- ฝึกเร็วและเหมาะกับ prototype 60–70%
- source code อ่านและพัฒนาต่อง่าย

Output tensor มี 3 channels ตามลำดับ: `surface`, `centerline`, `boundary`


In [ ]:
# ============================================================
# 13 — RESNET34 U-NET MULTI-TASK MODEL
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet34, ResNet34_Weights


class ConvBNAct(nn.Sequential):
    def __init__(self, in_ch, out_ch):
        super().__init__(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )


class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.conv = ConvBNAct(out_ch + skip_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))


class RoadResNet34UNet(nn.Module):
    ARCHITECTURE_VERSION = "road_resnet34_unet_centerline_v2"

    def __init__(self, in_channels=4, out_channels=3, pretrained=True):
        super().__init__()
        weights = ResNet34_Weights.DEFAULT if pretrained else None
        try:
            encoder = resnet34(weights=weights)
        except Exception as exc:
            print("ดาวน์โหลด ImageNet weights ไม่สำเร็จ; ใช้ random initialization แทน:", repr(exc))
            encoder = resnet34(weights=None)
            pretrained = False

        old_conv = encoder.conv1
        if in_channels != 3:
            new_conv = nn.Conv2d(
                in_channels, old_conv.out_channels,
                kernel_size=old_conv.kernel_size,
                stride=old_conv.stride,
                padding=old_conv.padding,
                bias=False,
            )
            with torch.no_grad():
                if pretrained:
                    new_conv.weight[:, :3] = old_conv.weight
                    if in_channels > 3:
                        mean_weight = old_conv.weight.mean(dim=1, keepdim=True)
                        for c in range(3, in_channels):
                            new_conv.weight[:, c:c+1] = mean_weight
                    elif in_channels < 3:
                        new_conv.weight[:] = old_conv.weight[:, :in_channels]
                else:
                    nn.init.kaiming_normal_(new_conv.weight, mode="fan_out", nonlinearity="relu")
            encoder.conv1 = new_conv

        self.stem = nn.Sequential(encoder.conv1, encoder.bn1, encoder.relu)   # /2, 64
        self.pool = encoder.maxpool
        self.enc1 = encoder.layer1   # /4, 64
        self.enc2 = encoder.layer2   # /8, 128
        self.enc3 = encoder.layer3   # /16, 256
        self.enc4 = encoder.layer4   # /32, 512

        self.dec4 = DecoderBlock(512, 256, 256)
        self.dec3 = DecoderBlock(256, 128, 128)
        self.dec2 = DecoderBlock(128, 64, 64)
        self.dec1 = DecoderBlock(64, 64, 64)
        self.final_refine = ConvBNAct(64, 32)
        self.head = nn.Conv2d(32, out_channels, kernel_size=1)

    def forward(self, x):
        input_size = x.shape[-2:]
        s0 = self.stem(x)
        s1 = self.enc1(self.pool(s0))
        s2 = self.enc2(s1)
        s3 = self.enc3(s2)
        b = self.enc4(s3)

        x = self.dec4(b, s3)
        x = self.dec3(x, s2)
        x = self.dec2(x, s1)
        x = self.dec1(x, s0)
        x = F.interpolate(x, size=input_size, mode="bilinear", align_corners=False)
        x = self.final_refine(x)
        return self.head(x)


model = RoadResNet34UNet(
    in_channels=len(BAND_INDEXES),
    out_channels=3,
    pretrained=USE_IMAGENET_PRETRAINED,
).to(DEVICE)

with torch.no_grad():
    test_logits = model(images[:1].to(DEVICE))
print("Model output:", tuple(test_logits.shape))
print("Architecture:", model.ARCHITECTURE_VERSION)
print("Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")


## Session 14 — Loss, Metrics, Optimizer และ Resume

แต่ละ output ใช้ BCE + soft Dice แยกกัน เพื่อแก้ class imbalance ของ centerline และ boundary

Validation score ที่ใช้เลือก checkpoint:

```text
0.60 × Surface Dice
+ 0.25 × Centerline Dice
+ 0.15 × Boundary Dice
```


In [ ]:

# ============================================================
# 14 — LOSS, METRICS, OPTIMIZER, AMP, RESUME (CENTERLINE V2)
# ============================================================

from scipy.ndimage import label as ndi_label

HEAD_NAMES = ["surface", "centerline", "boundary"]
EIGHT_CONNECTED = np.ones((3, 3), dtype=np.uint8)


class BinaryDiceLoss(nn.Module):
    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps = eps

    def forward(self, logits, target):
        prob = torch.sigmoid(logits)
        dims = (1, 2, 3)
        inter = (prob * target).sum(dims)
        denom = prob.sum(dims) + target.sum(dims)
        return 1.0 - ((2 * inter + self.eps) / (denom + self.eps)).mean()


class SoftFocalWithLogitsLoss(nn.Module):
    """Focal BCE that supports a [0, 1] Gaussian centreline target."""
    def __init__(self, alpha=CENTERLINE_FOCAL_ALPHA, gamma=CENTERLINE_FOCAL_GAMMA):
        super().__init__()
        self.alpha, self.gamma = float(alpha), float(gamma)

    def forward(self, logits, target):
        bce = F.binary_cross_entropy_with_logits(logits, target, reduction="none")
        prob = torch.sigmoid(logits)
        p_t = target * prob + (1.0 - target) * (1.0 - prob)
        alpha_t = target * self.alpha + (1.0 - target) * (1.0 - self.alpha)
        return (alpha_t * (1.0 - p_t).pow(self.gamma) * bce).mean()


dice_loss = BinaryDiceLoss()
centerline_focal_loss = SoftFocalWithLogitsLoss()


def head_loss(name, logits, target):
    if name == "centerline":
        return 0.70 * centerline_focal_loss(logits, target) + 0.30 * dice_loss(logits, target)
    bce = F.binary_cross_entropy_with_logits(logits, target)
    return 0.55 * bce + 0.45 * dice_loss(logits, target)


def total_loss_fn(logits, targets):
    losses, total = {}, 0.0
    for i, name in enumerate(HEAD_NAMES):
        value = head_loss(name, logits[:, i:i+1], targets[:, i:i+1])
        losses[name] = value
        total = total + HEAD_LOSS_WEIGHTS[name] * value
    return total, losses


def _mean_centerline_components(binary_prediction):
    # Diagnostic only: fewer components at comparable Dice indicates less fragmented lines.
    arr = binary_prediction.detach().cpu().numpy().astype(bool)
    counts = [ndi_label(tile[0], structure=EIGHT_CONNECTED)[1] for tile in arr]
    return float(np.mean(counts)) if counts else 0.0


@torch.no_grad()
def dice_iou_per_head(logits, targets, centerline_binary, threshold=0.5):
    probs = torch.sigmoid(logits)
    preds = probs >= threshold
    metrics = {}
    for i, name in enumerate(HEAD_NAMES):
        truth = centerline_binary[:, 0] >= 0.5 if name == "centerline" else targets[:, i] >= 0.5
        pred = preds[:, i]
        inter = (pred & truth).sum().float()
        union = (pred | truth).sum().float()
        denom = pred.sum().float() + truth.sum().float()
        metrics[f"{name}_iou"] = float((inter / (union + 1e-6)).cpu())
        metrics[f"{name}_dice"] = float((2 * inter / (denom + 1e-6)).cpu())
    center_pred = probs[:, 1:2] >= CENTERLINE_BINARY_METRIC_THRESHOLD
    metrics["centerline_components_per_tile"] = _mean_centerline_components(center_pred)
    return metrics


optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=3, min_lr=1e-6
)
try:
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
except TypeError:
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

START_EPOCH, BEST_SCORE, history = 1, -1.0, []
if RESUME_TRAINING:
    if not RESUME_CHECKPOINT.exists():
        raise FileNotFoundError(f"Resume checkpoint not found: {RESUME_CHECKPOINT}")
    ckpt = torch.load(RESUME_CHECKPOINT, map_location=DEVICE, weights_only=False)
    if ckpt.get("architecture_version") != model.ARCHITECTURE_VERSION:
        raise RuntimeError("Checkpoint ไม่ใช่ Centerline V2: ต้อง retrain จาก checkpoint ที่มี heatmap target")
    model.load_state_dict(ckpt["model_state_dict"], strict=True)
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    START_EPOCH = int(ckpt.get("epoch", 0)) + 1
    BEST_SCORE = float(ckpt.get("best_score", -1.0))
    history = list(ckpt.get("history", []))
    print("Resumed Centerline V2 from:", RESUME_CHECKPOINT)


## Session 15 — Training Loop และบันทึก Checkpoint

ระบบบันทึก:

- `last_theos2_road_multitask.pth` ทุก epoch
- `best_theos2_road_multitask.pth` เมื่อ validation score ดีขึ้น
- CSV log
- JSON configuration

Checkpoint เก็บ architecture version และ normalization configuration เพื่อให้ prediction notebook ตรวจความเข้ากันได้ก่อนโหลด


In [ ]:

# ============================================================
# 15 — TRAINING LOOP (CENTERLINE V2)
# ============================================================

import gc
import time
import json
import pandas as pd
from tqdm.auto import tqdm


def amp_context():
    try:
        return torch.amp.autocast("cuda", enabled=USE_AMP)
    except TypeError:
        return torch.cuda.amp.autocast(enabled=USE_AMP)


def run_epoch(loader, training=True):
    model.train(training)
    sums = {"loss": 0.0}
    for name in HEAD_NAMES:
        sums[f"{name}_iou"] = 0.0
        sums[f"{name}_dice"] = 0.0
    sums["centerline_components_per_tile"] = 0.0
    count = 0

    for batch_images, batch_targets, batch_centerline_binary in tqdm(loader, desc="Train" if training else "Val"):
        batch_images = batch_images.to(DEVICE, non_blocking=True)
        batch_targets = batch_targets.to(DEVICE, non_blocking=True)
        batch_centerline_binary = batch_centerline_binary.to(DEVICE, non_blocking=True)
        batch_size_now = batch_images.size(0)
        if training:
            optimizer.zero_grad(set_to_none=True)
            with amp_context():
                logits = model(batch_images)
                loss, _ = total_loss_fn(logits, batch_targets)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            with torch.no_grad(), amp_context():
                logits = model(batch_images)
                loss, _ = total_loss_fn(logits, batch_targets)

        metrics = dice_iou_per_head(logits.detach(), batch_targets, batch_centerline_binary)
        sums["loss"] += float(loss.item()) * batch_size_now
        for key, value in metrics.items():
            sums[key] += value * batch_size_now
        count += batch_size_now
    return {key: value / max(count, 1) for key, value in sums.items()}


def validation_score(metrics):
    return (
        0.45 * metrics["surface_dice"]
        + 0.40 * metrics["centerline_dice"]
        + 0.15 * metrics["boundary_dice"]
    )


def checkpoint_payload(epoch, train_metrics, val_metrics, best_score):
    return {
        "company": "",
        "task": "THEOS-2 road surface / centerline heatmap / boundary multi-task segmentation",
        "architecture_version": model.ARCHITECTURE_VERSION,
        "centerline_target_format": "surface-constrained Gaussian distance heatmap",
        "epoch": epoch, "best_score": best_score,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_metrics": train_metrics, "val_metrics": val_metrics, "history": history,
        "config": {
            "band_indexes": list(BAND_INDEXES), "head_names": HEAD_NAMES,
            "patch_size": PATCH_SIZE, "patch_stride": PATCH_STRIDE,
            "in_channels": len(BAND_INDEXES), "out_channels": 3,
            "normalization": "per-image percentile p02-p98",
            "centerline_heatmap_sigma_m": CENTERLINE_HEATMAP_SIGMA_M,
            "centerline_heatmap_radius_m": CENTERLINE_HEATMAP_RADIUS_M,
            "centerline_binary_width_m": CENTERLINE_BINARY_WIDTH_M,
            "centerline_metric_threshold": CENTERLINE_BINARY_METRIC_THRESHOLD,
        },
    }


LAST_CKPT = CHECKPOINT_DIR / "last_theos2_road_multitask_centerline_v2.pth"
BEST_CKPT = CHECKPOINT_DIR / "best_theos2_road_multitask_centerline_v2.pth"
start_time = time.time()
for epoch in range(START_EPOCH, EPOCHS + 1):
    print(f"\n{'=' * 100}\nEpoch {epoch}/{EPOCHS}\n{'=' * 100}")
    train_metrics = run_epoch(train_loader, training=True)
    val_metrics = run_epoch(val_loader, training=False)
    score = validation_score(val_metrics)
    scheduler.step(score)

    row = {"epoch": epoch, "lr": optimizer.param_groups[0]["lr"], "train_loss": train_metrics["loss"], "val_loss": val_metrics["loss"], "val_score": score}
    for key, value in train_metrics.items():
        if key != "loss": row[f"train_{key}"] = value
    for key, value in val_metrics.items():
        if key != "loss": row[f"val_{key}"] = value
    history.append(row)
    print(f"Train loss {train_metrics['loss']:.4f} | Val loss {val_metrics['loss']:.4f} | Score {score:.4f}")
    print(f"Surface Dice {val_metrics['surface_dice']:.4f} | Center Dice {val_metrics['centerline_dice']:.4f} | Boundary Dice {val_metrics['boundary_dice']:.4f} | Center components/tile {val_metrics['centerline_components_per_tile']:.2f}")

    torch.save(checkpoint_payload(epoch, train_metrics, val_metrics, max(BEST_SCORE, score)), LAST_CKPT)
    if score > BEST_SCORE:
        BEST_SCORE = score
        torch.save(checkpoint_payload(epoch, train_metrics, val_metrics, BEST_SCORE), BEST_CKPT)
        print("Saved new best:", BEST_CKPT)
    pd.DataFrame(history).to_csv(LOG_DIR / "road_multitask_training_log_centerline_v2.csv", index=False)
    gc.collect(); torch.cuda.empty_cache()

elapsed_minutes = (time.time() - start_time) / 60.0
run_config = {"manifest": str(manifest_path), "best_checkpoint": str(BEST_CKPT), "last_checkpoint": str(LAST_CKPT), "epochs": EPOCHS, "best_score": BEST_SCORE, "elapsed_minutes": elapsed_minutes}
with open(LOG_DIR / "road_multitask_run_config_centerline_v2.json", "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2, ensure_ascii=False)
print("Training finished | Best checkpoint:", BEST_CKPT)
display(pd.DataFrame(history))


## Session 16 — ตรวจ Prediction ตัวอย่างจาก Validation Set

ภาพนี้ใช้ตรวจอย่างรวดเร็วว่า model ไม่ได้เรียนเป็น mask ว่างทั้งหมด และ output ทั้ง 3 channel มีรูปแบบสมเหตุสมผล


In [ ]:

# ============================================================
# 16 — VALIDATION PREVIEW (CENTERLINE V2)
# ============================================================

import matplotlib.pyplot as plt

best = torch.load(BEST_CKPT, map_location=DEVICE, weights_only=False)
if best.get("architecture_version") != model.ARCHITECTURE_VERSION:
    raise RuntimeError("Loaded checkpoint is not Centerline V2")
model.load_state_dict(best["model_state_dict"], strict=True)
model.eval()

sample_images, sample_targets, sample_center_binary = next(iter(val_loader))
with torch.no_grad(), amp_context():
    sample_probs = torch.sigmoid(model(sample_images.to(DEVICE))).cpu().numpy()

n_show = min(3, len(sample_images))
for i in range(n_show):
    rgb = np.transpose(sample_images[i, :3].numpy(), (1, 2, 0))
    target_heatmap = sample_targets[i, 1].numpy()
    metric_binary = sample_center_binary[i, 0].numpy() > 0.5
    pred_heatmap = sample_probs[i, 1]
    pred_binary = pred_heatmap >= CENTERLINE_BINARY_METRIC_THRESHOLD
    overlay = rgb.copy()
    overlay[pred_binary] = np.array([0.0, 1.0, 1.0])
    overlay[metric_binary] = 0.5 * overlay[metric_binary] + 0.5 * np.array([1.0, 1.0, 0.0])

    fig, axes = plt.subplots(2, 4, figsize=(18, 9))
    axes[0, 0].imshow(rgb); axes[0, 0].set_title("RGB")
    axes[0, 1].imshow(sample_targets[i, 0], cmap="gray"); axes[0, 1].set_title("Label: surface")
    axes[0, 2].imshow(target_heatmap, cmap="magma", vmin=0, vmax=1); axes[0, 2].set_title("Target: centerline heatmap")
    axes[0, 3].imshow(metric_binary, cmap="gray"); axes[0, 3].set_title("Metric label: binary centerline")
    axes[1, 0].imshow(rgb); axes[1, 0].imshow(sample_probs[i, 0], cmap="spring", alpha=0.35, vmin=0, vmax=1); axes[1, 0].set_title("Surface probability")
    axes[1, 1].imshow(pred_heatmap, cmap="magma", vmin=0, vmax=1); axes[1, 1].set_title("Prediction: centerline heatmap")
    axes[1, 2].imshow(pred_binary, cmap="gray"); axes[1, 2].set_title(f"Prediction binary ≥ {CENTERLINE_BINARY_METRIC_THRESHOLD}")
    axes[1, 3].imshow(overlay); axes[1, 3].set_title("Cyan prediction / yellow target")
    for ax in axes.ravel(): ax.axis("off")
    plt.tight_layout()
    out = PREVIEW_DIR / f"validation_prediction_centerline_v2_{i+1}.png"
    plt.savefig(out, dpi=160, bbox_inches="tight")
    plt.show()
    print("Saved:", out)


## Session 17 — สรุปไฟล์ Output

ไฟล์สำคัญที่สุดสำหรับ prediction notebook คือ:

```text
best_theos2_road_multitask.pth
```

Weak labels, manifest, logs และ previews ถูกเก็บใน Project Root บน Google Drive แล้ว จึงยังอยู่แม้ Colab Runtime ถูกลบ


In [ ]:
# ============================================================
# 17 — OUTPUT SUMMARY
# ============================================================

print("PROJECT ROOT:", PROJECT_ROOT)
print("\nBest checkpoint:", BEST_CKPT)
print("Last checkpoint:", LAST_CKPT)
print("Manifest:", manifest_path)
print("Training log:", LOG_DIR / "road_multitask_training_log.csv")

print("\nFiles created under project root:")
for p in sorted(PROJECT_ROOT.rglob("*")):
    if p.is_file():
        print(f" - {p.relative_to(PROJECT_ROOT)} | {p.stat().st_size / 1024**2:.2f} MB")


## เอกสารอ้างอิงทางเทคนิค

- OSMnx documentation: https://osmnx.readthedocs.io/en/stable/
- Torchvision ResNet34: https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet34.html
- Rasterio vector features: https://rasterio.readthedocs.io/en/stable/topics/features.html

โน้ตบุ๊กนี้ตั้งใจเป็น foundation code; เมื่อมี label ที่แม่นยำกว่า สามารถแทน weak labels เดิมแล้วฝึกใหม่โดยไม่ต้องเปลี่ยน model/prediction pipeline หลัก
